In [ ]:
# =========================================================
# JUPYTER NOTEBOOK: Milvus Speaker Embedding & Validation
# =========================================================

# Cell 1: Imports
import os
from datetime import datetime

import torch
import torchaudio
from torchaudio.transforms import Resample
from speechbrain.inference.speaker import EncoderClassifier

from pymilvus import (
    connections,
    Collection,
    FieldSchema,
    CollectionSchema,
    DataType,
    utility
)

import numpy as np

print("📦 Imports loaded")


In [ ]:
# Cell 2: CONFIGURATION
DEVICE = "cpu"  # or "cuda" if available
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"
COLLECTION_NAME = "agent_voice_embeddings"

# Add multiple reference audios per agent
AGENT_AUDIO_MAP = {
    #"Nikhil": [
     #   "/home/jignesh/AI/Transcribe/Latest/ReferenceData/NikhilData/Trimmed_recording_Xchangedd35f588-9b2e-45fa-b453-6af327e5802d.mp3"
  #  ],
    #"Chaitali": [
       # "/home/jignesh/AI/Transcribe/AudioFiles/Trimmed_recording_Xchangeae720826-793e-49bc-b006-e337ba4d6cb0.wav",
      #  "/home/jignesh/AI/Transcribe/Latest/Trimmed_recording_Xchangeae720826-793e-49bc-b006-e337ba4d6cb0 (1).wav"
    #]
   # "Siddharth Indamdar": [
   #     "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_092537_9987063449_XX9443992881_OUT (1).mp3"
  #  ],
    "Aditi Joshi": [
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_092708_9136125828_XX8069199888_IN (1).mp3"
    ],
    "Suraj Jadhav":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/idea-09032026-093025-8657040811-xx9849164040-out_wfZtOi1X.mp3"
    ],
    "Pragya Upadhya":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_093922_8828511415_XX9342259514_IN (1).mp3"
    ],
    "Ashutosh Vyas":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_094229_9987067710_XX9442704846_OUT (1).mp3"
    ],
    "Darshan Shelke":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_095024_8976968981_XX8610221289_OUT (1).mp3"
    ],
    "Rohit Jain":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_100258_9967582763_XX7904891287_OUT (1).mp3"
    ],
    "Sanjana Prajapati":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_101444_8655802643_XX8069199886_IN (1).mp3"
    ],
    "Priya Verma":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_110759_8097504582_XX7904295203_OUT (1).mp3"
    ],
    "Shankar Saini":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_112035_8828838400_XX9974341519_OUT (1).mp3"
    ],
    "Divyansh Dharam":[
        "/home/jignesh/AI/Transcribe/Latest/AllAgentsReferenceAudio/IDEA_09032026_124658_9004486398_X09773007559_OUT (1).mp3"
    ]

}

VALIDATION_THRESHOLD = 0.65  # cosine similarity threshold for accepting a reference

print("⚙️ Configuration loaded")


In [ ]:
from pymilvus import connections

connections.connect(
    alias="default",
    host="localhost",
    port="19530"
)

print("✅ Connected to Milvus")

In [ ]:
# ==============================
# 3️⃣ CREATE COLLECTION (IF NOT EXISTS)
# ==============================
if not utility.has_collection(COLLECTION_NAME):
    fields = [
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="agent_name", dtype=DataType.VARCHAR, max_length=100),
        FieldSchema(name="audio_path", dtype=DataType.VARCHAR, max_length=500),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=192)
    ]

    schema = CollectionSchema(fields=fields, description="Agent voice reference embeddings")
    collection = Collection(name=COLLECTION_NAME, schema=schema)

    # Create index for fast similarity search
    index_params = {
        "metric_type": "COSINE",
        "index_type": "HNSW",
        "params": {"M": 16, "efConstruction": 200}
    }
    collection.create_index(field_name="embedding", index_params=index_params)
    print(f"✅ Collection '{COLLECTION_NAME}' created.")
else:
    collection = Collection(COLLECTION_NAME)
    print(f"ℹ️ Collection '{COLLECTION_NAME}' already exists.")

collection.load()

In [ ]:
# ==============================
# 4️⃣ LOAD SPEAKER MODEL
# ==============================
print("Loading speaker model...")
speaker_encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": DEVICE}
)

In [ ]:
# ==============================
# 5️⃣ AUDIO LOADER FUNCTION
# ==============================
def load_audio(path):
    waveform, sr = torchaudio.load(path)
    
    # Mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    # Resample to 16kHz
    if sr != 16000:
        waveform = Resample(orig_freq=sr, new_freq=16000)(waveform)
    
    return waveform

In [ ]:
# ==============================
# 6️⃣ EMBEDDING GENERATOR
# ==============================
def generate_embedding(audio_path):
    waveform = load_audio(audio_path)
    with torch.no_grad():
        emb = speaker_encoder.encode_batch(waveform).squeeze()
    return emb.cpu().tolist()

In [ ]:
# ==============================
# 7️⃣ INSERT FUNCTION
# ==============================
def insert_reference(agent_name, audio_path):
    print("\n📥 Adding reference audio:")
    print(f"   Agent : {agent_name}")
    print(f"   File  : {audio_path}")
    
    # Generate embedding
    embedding = generate_embedding(audio_path)
    
    # Insert into Milvus
    collection.insert([
        [agent_name],      # agent_name
        [audio_path],      # audio_path
        [embedding]        # embedding
    ])
    collection.flush()
    
    print(f"✅ Inserted successfully.")

In [ ]:
# ==============================
# 8️⃣ ADD ALL REFERENCE AUDIOS
# ==============================
for agent, audio_list in AGENT_AUDIO_MAP.items():
    for audio_path in audio_list:
        insert_reference(agent, audio_path)

# ==============================
# 9️⃣ SUMMARY
# ==============================
print("\n🎉 DONE")
print(f"🧠 Total embeddings in collection: {collection.num_entities}")



